# Docker Fundamentals — Azure Supplement

> **Mission:** Deploy Flask app to Azure Container Registry and Azure Container Instances.
>
> **This notebook:** Azure-specific workflows (ACR, ACI, monitoring). Requires Azure subscription (free tier available).
>
> **Main notebook:** [notebook.ipynb](notebook.ipynb) — 100% local Docker workflow (no cloud dependencies).

---

## Prerequisites

- **Azure subscription** ([free account](https://azure.microsoft.com/free/) — $200 credit + 12 months free services)
- **Azure CLI** installed ([install guide](https://learn.microsoft.com/en-us/cli/azure/install-azure-cli))
- **Docker Desktop** running locally
- **Completed main notebook** (built `flask-app:v1` image)

Verify Azure CLI:
```bash
az version
# "azure-cli": "2.60.0" or later
```

---

## Cell 1: Azure Credentials and Configuration

**Goal:** Configure Azure credentials and set resource names.

**Security:**
- Credentials are **stripped by pre-push hook** (see `scripts/install-hooks.ps1`)
- Never commit real subscription IDs or secrets
- Use empty strings as placeholders

**What we're configuring:**
- Azure subscription ID
- Resource group name
- Container registry name (must be globally unique)
- Region

In [ ]:
# TODO: Implement this cell


**Expected output:**
```
Azure Configuration:
  Subscription ID: 12345678-abcd-1234-abcd-123456789abc
  Resource Group: rg-devops-docker-demo
  Registry Name: acra3f5d8e7
  Location: eastus
  Image: flask-app:v1

✅ Logged in as: user@example.com
   Active subscription: Visual Studio Enterprise

✅ Configuration valid. Ready to create Azure resources.
```

**If not logged in:**
```bash
az login
# Opens browser for authentication
```

**Registry naming rules:**
- 5-50 characters
- Lowercase letters and numbers only
- Must be globally unique across all Azure customers

✅ **Checkpoint:** Azure credentials configured. Next: create Container Registry.

---

## Cell 2: Create Azure Container Registry (ACR)

**Goal:** Create Azure Container Registry to host Docker images.

**What we're doing:**
1. Create resource group
2. Create Container Registry (Basic SKU — free tier for 12 months)
3. Enable admin credentials (for Docker login)
4. Retrieve registry credentials

**ACR SKUs:**
- **Basic** — 10 GB storage, 10 webhooks, good for dev/test ($0.17/day)
- **Standard** — 100 GB storage, geo-replication
- **Premium** — 500 GB storage, content trust, private endpoints

In [ ]:
# TODO: Implement this cell


**Expected output:**
```
Creating resource group: rg-devops-docker-demo
{
  "location": "eastus",
  "name": "rg-devops-docker-demo",
  "properties": {
    "provisioningState": "Succeeded"
  }
}

Creating Container Registry: acra3f5d8e7
{
  "adminUserEnabled": true,
  "creationDate": "2026-04-26T10:30:00.000000+00:00",
  "loginServer": "acra3f5d8e7.azurecr.io",
  "name": "acra3f5d8e7",
  "provisioningState": "Succeeded",
  "sku": "Basic"
}

Registry details:
  Login server: acra3f5d8e7.azurecr.io

Admin credentials:
Username       Password
-------------- --------------------------------
acra3f5d8e7    Abc123...xyz789==

✅ Container Registry created successfully.
   Login server: acra3f5d8e7.azurecr.io
```

**Cost estimate (Basic SKU):**
- Storage: $0.10 per GB/month (first 10 GB included)
- Data egress: First 100 GB free per month
- **Total for this demo: ~$5/month** (delete after testing to avoid charges)

**Security note:** Admin credentials are shown here for simplicity. In production, use:
- **Service principals** for CI/CD pipelines
- **Managed identities** for Azure services (AKS, ACI)
- **Azure AD tokens** for user authentication

✅ **Checkpoint:** ACR created and accessible. Next: build and push image.

---

## Cell 3: Build and Push Image to ACR

**Goal:** Build Docker image and push to Azure Container Registry.

**What we're doing:**
1. Login to ACR using Azure CLI
2. Tag local image with ACR login server
3. Push image to ACR
4. Verify image in registry

**Alternative approach:** Use `az acr build` to build directly in Azure (no local Docker required).

In [ ]:
# TODO: Implement this cell


**Expected output:**
```
Login server: acra3f5d8e7.azurecr.io

Logging in to ACR...
Login Succeeded

Tagging image: flask-app:v1 -> acra3f5d8e7.azurecr.io/flask-app:v1

Pushing images to ACR...
The push refers to repository [acra3f5d8e7.azurecr.io/flask-app]
v1: digest: sha256:a3f5d8e7c2b1... size: 1234
latest: digest: sha256:a3f5d8e7c2b1... size: 1234

Images in ACR:
Result
----------
flask-app

Image tags:
Result
--------
latest
v1

✅ Image pushed successfully.
   Full image path: acra3f5d8e7.azurecr.io/flask-app:v1
```

**Alternative: Build in Azure (no local Docker):**
```bash
cd flask-docker-app
az acr build \
  --registry $REGISTRY_NAME \
  --image flask-app:v1 \
  --file Dockerfile \
  .
```

**Benefits of `az acr build`:**
- No local Docker required (build happens in Azure)
- Automatically pushes to registry
- Build logs stored in Azure
- Useful for CI/CD pipelines

✅ **Checkpoint:** Image is now in ACR. Next: deploy to Azure Container Instances.

---

## Cell 4: Deploy to Azure Container Instances (ACI)

**Goal:** Run Flask container on Azure Container Instances.

**What we're doing:**
1. Get ACR credentials
2. Create container instance with ACR image
3. Expose port 5000 with public IP
4. Test the deployed app

**ACI use cases:**
- Quick testing/demos (no infrastructure setup)
- Batch jobs (run once and terminate)
- CI/CD ephemeral environments
- **Not for production web apps** (use App Service or AKS for high availability)

In [ ]:
# TODO: Implement this cell


**Expected output:**
```
Creating container instance: flask-aci-demo
  Image: acra3f5d8e7.azurecr.io/flask-app:v1
  CPU: 1 core, Memory: 1.5 GB

{
  "provisioningState": "Succeeded",
  "ipAddress": {
    "fqdn": "flask-demo-a3f5d8e7.eastus.azurecontainer.io",
    "ip": "20.42.123.45",
    "ports": [{"port": 5000, "protocol": "TCP"}]
  }
}

Container details:
FQDN                                           IP             State
--------------------------------------------   -------------  -------
flask-demo-a3f5d8e7.eastus.azurecontainer.io  20.42.123.45   Running

Waiting 10 seconds for container to start...

Testing deployed app:
{
    "flask": "ok",
    "redis": "unavailable"
}

✅ Container deployed successfully.
   URL: http://flask-demo-a3f5d8e7.eastus.azurecontainer.io:5000
```

**Note:** Redis is unavailable because we only deployed Flask (ACI doesn't support Docker Compose). For multi-container deployments, use:
- **Azure Container Apps** (managed Kubernetes with automatic scaling)
- **Azure Kubernetes Service (AKS)** (full Kubernetes control)

**Cost estimate (ACI):**
- 1 vCPU + 1.5 GB RAM: ~$0.0000125/second = $0.045/hour = $1.08/day
- **Stop container when not in use:** `az container stop --name $CONTAINER_NAME`

✅ **Checkpoint:** Flask app deployed and publicly accessible. Next: monitoring and logs.

---

## Cell 5: Monitor Container with Azure Container Insights

**Goal:** View logs and metrics for deployed container.

**What we're doing:**
1. Stream container logs (stdout/stderr)
2. Check container metrics (CPU, memory)
3. View events (restarts, errors)
4. Cleanup resources

**Production monitoring:** For production workloads, enable **Container Insights** (Azure Monitor integration) for advanced metrics, alerting, and log aggregation.

In [ ]:
# TODO: Implement this cell


**Expected output:**
```
Container logs (last 20 lines):
 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
127.0.0.1 - - [26/Apr/2026 11:00:05] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [26/Apr/2026 11:00:10] "GET / HTTP/1.1" 200 -

Container state and events:
State: Running
Restart: 0
Events:
  - Type: Normal
    Message: Successfully pulled image "acra3f5d8e7.azurecr.io/flask-app:v1"
  - Type: Normal
    Message: Started container

---

🧹 Cleanup (delete all resources to avoid charges):

Delete container instance:
  az container delete --resource-group rg-devops-docker-demo --name flask-aci-demo --yes

Delete entire resource group (ACR + ACI):
  az group delete --name rg-devops-docker-demo --yes --no-wait

⚠️  Resource group deletion is PERMANENT and cannot be undone.
```

**Advanced monitoring with Container Insights:**

1. **Enable Log Analytics workspace:**
   ```bash
   az monitor log-analytics workspace create \
     --resource-group $RESOURCE_GROUP \
     --workspace-name aci-logs
   ```

2. **Create container with diagnostics:**
   ```bash
   az container create \
     --log-analytics-workspace <workspace-id> \
     --log-analytics-workspace-key <workspace-key>
   ```

3. **View in Azure Portal:**
   - Container Instances → Monitoring → Logs
   - Query with Kusto (KQL): `ContainerInstanceLog_CL | limit 100`

**Cost estimate (Container Insights):**
- Log Analytics: $2.30 per GB ingested (first 5 GB/month free)
- Typical Flask app: ~50 MB logs/day = **$3.45/month**

✅ **Checkpoint:** Container is monitored and ready for cleanup. Run the cleanup commands to avoid charges.

---

## Summary

✅ **What you deployed:**
- Azure Container Registry (private Docker registry)
- Docker image pushed to ACR
- Flask app running on Azure Container Instances
- Public FQDN with HTTP access

✅ **Key Azure concepts:**
- ACR for private image hosting
- ACI for serverless container execution
- Azure CLI for infrastructure management
- Container logs and monitoring

**Next steps:**
- [Ch.2 — Container Orchestration](../02_container_orchestration) — Docker Compose for multi-container apps
- [Ch.3 — Kubernetes Basics](../03_kubernetes_basics) — Deploy to AKS (Azure Kubernetes Service)

**Production alternatives to ACI:**
- **Azure Container Apps** — Managed Kubernetes with auto-scaling, ingress, Dapr integration
- **Azure Kubernetes Service (AKS)** — Full Kubernetes cluster for complex microservices
- **Azure App Service** — PaaS for web apps (supports Docker containers)